In [2]:
import pandas as pd 
import numpy as np 
%run 02_retention_analysis.ipynb


Avg LTV (repeat): $277.30
Lost repeat customers: 41
Revenue at Risk: $11,467.03
Total Customers: 96,096
Current Repeat Rate: 3.1%
Current Repeat Customers: 2,978
Target Repeat Customers (15%): 14,414
Retention Gap: 11,436
Avg Repeat Customer LTV: $277.30
Revenue Opportunity: $3,171,202.80


In [4]:
order_level = pd.read_csv("/home/user/Suraj/Work/E_Commerce/Cleaned/ready/order_level.csv")
customer_level = pd.read_csv("/home/user/Suraj/Work/E_Commerce/Cleaned/ready/customer_level.csv")
seller_level = pd.read_csv("/home/user/Suraj/Work/E_Commerce/Cleaned/ready/seller_level.csv")

### Seller Accountability

In [5]:
seller_level

,Unnamed: 0,seller_id,total_orders,total_revenue,delayed_orders,avg_delivery_delay,avg_review_score,one_star_pct,delay_pct
0,0,0015a82c2db000af6aaaf3ae2ecb0532,3,2748.06,0,-16.333333,3.666667,33.333333,0.000000
1,1,001cca7ae9ae17fb1caed9dfb1094831,200,33934.17,12,-12.882051,3.984772,12.000000,6.000000
2,2,001e6ad469a905060d959994f1b41e4f,1,267.94,0,NaN,1.000000,100.000000,0.000000
3,3,002100f778ceb8431b7a1020ff7ab48f,51,2028.16,9,-8.180000,3.901961,13.725490,17.647059
4,4,003554e2dce176b5555353e4f3555ac8,1,139.38,0,-27.000000,5.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
3090,3090,ffcfefa19b08742c5d315f2791395ee5,1,79.52,0,NaN,1.000000,100.000000,0.000000
3091,3091,ffdd9f82b9a447f6f8d4b91554cc7dd3,18,2828.66,0,-12.388889,4.333333,5.555556,0.000000
3092,3092,ffeee66ac5d5a62fe688b9d26f83f534,14,2259.55,2,-8.357143,4.214286,14.285714,14.285714
3093,3093,fffd5413c0700ac820c7069d66d98c89,60,11896.04,7,-11.228070,3.847458,18.333333,11.666667


In [6]:
revenue_threshold = seller_level['total_revenue'].median()
reviews_threshold = seller_level['avg_review_score'].median()
print(revenue_threshold)
print(reviews_threshold)

996.84
4.185699746751556


In [7]:
high_rev = seller_level['total_revenue']>=revenue_threshold
low_rev = ~high_rev
good_score = seller_level['avg_review_score']>= reviews_threshold
bad_score = ~good_score

In [8]:
conditions = [ (high_rev & good_score), (high_rev & bad_score) ,(low_rev & good_score) ,(low_rev & bad_score)]
labels = ["Stars","Risky Cash Cows","Hidden Gems","Deadwight"]
seller_level['segments']=np.select(conditions,labels,default="Other")

In [9]:
seller_level['segments'].value_counts()

segments
Risky Cash Cows    822
Hidden Gems        819
Deadwight          728
Stars              726
Name: count, dtype: int64

In [10]:
total_revenue_demo = seller_level['total_revenue'].sum()
segment_summary = seller_level.groupby('segments').agg(
    total_sellers=('seller_id','count'),
    total_revenue=('total_revenue', 'sum'),
    average_delay_rate=('delay_pct', 'mean'),
    average_score=('avg_review_score', 'mean')
)
segment_summary['pct_revenue'] = round((segment_summary['total_revenue']/total_revenue_demo)*100,2)
segment_summary['one_star_rating'] = seller_level.groupby('segments')['one_star_pct'].mean()

In [11]:
segment_summary

,total_sellers,total_revenue,average_delay_rate,average_score,pct_revenue,one_star_rating
segments,,,,,,
Deadwight,728,262573.87,10.357160,2.943433,1.66,34.749897
Hidden Gems,819,285434.73,1.590517,4.797414,1.80,0.775482
Risky Cash Cows,822,9050113.22,9.267638,3.710473,57.12,19.307327
Stars,726,6245431.42,3.895623,4.476582,39.42,4.507801


In [12]:
total_revenue_demo = seller_level['total_revenue'].sum()
segment_summary = seller_level.groupby('segments').agg(
    total_sellers=('seller_id','count'),
    total_revenue=('total_revenue', 'sum'),
    average_delay_rate=('delay_pct', 'mean'),
    average_score=('avg_review_score', 'mean')
)
segment_summary['pct_revenue'] = round((segment_summary['total_revenue']/total_revenue_demo)*100,2)
segment_summary['one_star_rating'] = seller_level.groupby('segments')['one_star_pct'].mean()

In [13]:
total_deadweight_orders=seller_level['total_orders'].where(seller_level['segments']=='Deadwight').sum()

In [14]:
lost_customers = total_deadweight_orders * 0.347

In [15]:
avg_lost_ltv = lost_customers* avg_ltv
avg_lost_ltv

np.float64(242482.212)

In [16]:
# Hardcode the revenue value
deadweight_revenue = 262574

# Recalculate net impact
lost_ltv = 242482.21
net_impact = lost_ltv - deadweight_revenue

print(f"H. Net Impact: ${net_impact:,.2f}")

H. Net Impact: $-20,091.79


In [17]:
# Risky Cash Cows stats
rcc_orders = seller_level[seller_level['segments'] == 'Risky Cash Cows']['total_orders'].sum()
rcc_one_star_rate = 0.193  # 19.3%
rcc_revenue = 9050113

# Ruined customers from Risky Cash Cows
rcc_ruined = rcc_orders * rcc_one_star_rate

# Lost LTV
rcc_lost_ltv = rcc_ruined * 277.30

print(f"Risky Cash Cow orders: {rcc_orders:,}")
print(f"Ruined customers: {rcc_ruined:,.0f}")
print(f"Lost LTV from Risky Cash Cows: ${rcc_lost_ltv:,.2f}")
print(f"Revenue from Risky Cash Cows: ${rcc_revenue:,.2f}")

Risky Cash Cow orders: 59,640
Ruined customers: 11,511
Lost LTV from Risky Cash Cows: $3,191,867.20
Revenue from Risky Cash Cows: $9,050,113.00
